
# Word2Vec: Learning Distributed Word Representations

**Deep Learning – B.Sc. Computer Science**

This notebook introduces Word2Vec from a modern deep-learning perspective. The emphasis is on:

- Understanding **distributed representations**
- Understanding how **training samples are constructed**
- Relating Word2Vec to the **Encoder–Decoder** idea discussed in the Autoencoders lecture
- Training and using Word2Vec with **Gensim**
- Exploring pre-trained embeddings with **spaCy**

We intentionally avoid implementing Word2Vec from scratch. In practice, students should understand the data flow, architecture, and learned representations rather than low-level implementation details.



# 1. Why Do We Need Word Embeddings?

Traditional one-hot vectors suffer from three major problems:

1. Very high dimensionality
2. Sparse representations
3. No notion of semantic similarity

For example, using one-hot encoding:

- `cat`
- `dog`
- `computer`

are all equally distant from each other.

Word embeddings solve this problem by learning dense vectors in which semantically related words tend to appear close together.



# 2. Distributional Hypothesis

The central idea behind Word2Vec is:

> You shall know a word by the company it keeps.

Words appearing in similar contexts tend to have similar meanings.

Examples:

- "The **cat** chased the mouse."
- "The **dog** chased the ball."

Since **cat** and **dog** occur in similar contexts, their vectors should become similar during training.



# 3. Word2Vec as an Encoder–Decoder Model

Recall the encoder–decoder idea from the Autoencoders lecture.

## Autoencoder

Input → Encoder → Latent Representation → Decoder → Reconstructed Input

The hidden representation is the important part because it compresses useful information.

## Word2Vec

Word2Vec follows a very similar philosophy:

### Skip-Gram

Center Word → Encoder (Embedding Layer) → Decoder → Context Words

### CBOW

Context Words → Encoder (Embedding Layer) → Decoder → Center Word

The embedding layer acts as a **latent representation space**.

Just as an autoencoder learns useful compressed features for images, Word2Vec learns useful compressed features for words.

After training, we discard the decoder and keep only the learned embeddings.



# 4. Skip-Gram and CBOW

## Skip-Gram

Given a center word, predict nearby words.

Sentence:

> Anne was very happy today

Using a window size of 2:

Center word:

`very`

Context words:

`Anne`, `was`, `happy`, `today`

Training pairs:

- (very → Anne)
- (very → was)
- (very → happy)
- (very → today)

---

## CBOW

Reverse the task.

Input:

`Anne`, `was`, `happy`, `today`

Target:

`very`


In [1]:

sentence = "Anne was very happy today".split()

window_size = 2

for i, center in enumerate(sentence):
    context = []
    for j in range(max(0, i-window_size), min(len(sentence), i+window_size+1)):
        if i != j:
            context.append(sentence[j])
    print(f"Center: {center:>5} --> Context: {context}")


Center:  Anne --> Context: ['was', 'very']
Center:   was --> Context: ['Anne', 'very', 'happy']
Center:  very --> Context: ['Anne', 'was', 'happy', 'today']
Center: happy --> Context: ['was', 'very', 'today']
Center: today --> Context: ['very', 'happy']



# 5. Preparing Training Samples

The most important practical step in Word2Vec is creating training examples.

Suppose we have:

> Anne was sitting by the window reading a book

With a window size of 2:

| Center Word | Context Word |
|------------|-------------|
| sitting | was |
| sitting | by |
| sitting | Anne |
| sitting | the |
| window | by |
| window | the |
| window | reading |
| window | a |

These center-context pairs become the supervised training data.


In [2]:

def generate_skipgram_pairs(tokens, window=2):
    pairs = []

    for i, center in enumerate(tokens):
        start = max(0, i-window)
        end = min(len(tokens), i+window+1)

        for j in range(start, end):
            if i != j:
                pairs.append((center, tokens[j]))

    return pairs


tokens = "Anne was sitting by the window reading a book".split()

pairs = generate_skipgram_pairs(tokens, window=2)

pairs[:20]


[('Anne', 'was'),
 ('Anne', 'sitting'),
 ('was', 'Anne'),
 ('was', 'sitting'),
 ('was', 'by'),
 ('sitting', 'Anne'),
 ('sitting', 'was'),
 ('sitting', 'by'),
 ('sitting', 'the'),
 ('by', 'was'),
 ('by', 'sitting'),
 ('by', 'the'),
 ('by', 'window'),
 ('the', 'sitting'),
 ('the', 'by'),
 ('the', 'window'),
 ('the', 'reading'),
 ('window', 'by'),
 ('window', 'the'),
 ('window', 'reading')]


# 6. Working with *Anne of Green Gables*

The course dataset is expected at:

```text
data/Anne_of_Green_Gables.txt
```

The following code loads and preprocesses the corpus.


In [3]:

from pathlib import Path
import re

text = Path("data/Anne_of_Green_Gables.txt").read_text(encoding="utf-8")

text = text.lower()
text = re.sub(r"[^a-z\s]", " ", text)

tokens = text.split()

print("Number of tokens:", len(tokens))
print(tokens[:50])


Number of tokens: 107178
['title', 'anne', 'of', 'green', 'gables', 'author', 'lucy', 'maud', 'montgomery', 'release', 'date', 'ebook', 'last', 'updated', 'october', 'language', 'english', 'anne', 'of', 'green', 'gables', 'by', 'lucy', 'maud', 'montgomery', 'table', 'of', 'contents', 'chapter', 'i', 'mrs', 'rachel', 'lynde', 'is', 'surprised', 'chapter', 'ii', 'matthew', 'cuthbert', 'is', 'surprised', 'chapter', 'iii', 'marilla', 'cuthbert', 'is', 'surprised', 'chapter', 'iv', 'morning']



# 7. Training Word2Vec with Gensim

We use the industrial-strength implementation provided by Gensim.


In [4]:

from gensim.models import Word2Vec

sentences = [tokens[i:i+50] for i in range(0, len(tokens), 50)]

model = Word2Vec(
    sentences,
    vector_size=100,
    window=5,
    min_count=5,
    workers=4,
    sg=1
)

model.train(
    sentences,
    total_examples=len(sentences),
    epochs=10
)


c:\Users\m.amintoosi\.conda\envs\pth-gpu\lib\site-packages\google\api_core\_python_version_support.py:263: FutureWarning: You are using a Python version (3.10.16) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


(671857, 1071780)


# 8. Exploring the Learned Embeddings


In [5]:

model.wv.most_similar("anne", topn=10)


[('shortly', 0.6128869652748108),
 ('whatever', 0.6066677570343018),
 ('exclaimed', 0.5910108089447021),
 ('vexed', 0.5849132537841797),
 ('wistfully', 0.5798739194869995),
 ('seriously', 0.5781537294387817),
 ('solemnly', 0.5772078037261963),
 ('gravely', 0.5765380263328552),
 ('bitterly', 0.5700002908706665),
 ('indignantly', 0.568381130695343)]

In [6]:

model.wv.similarity("anne", "marilla")


0.55376774

In [7]:

model.wv.most_similar(
    positive=["woman", "king"],
    negative=["man"],
    topn=10
)


KeyError: "Key 'king' not present in vocabulary"


# 9. Visualizing Embeddings

High-dimensional embeddings can be projected into two dimensions using t-SNE.


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

words = [
    "anne",
    "marilla",
    "matthew",
    "school",
    "house",
    "girl",
    "friend"
]

words = [w for w in words if w in model.wv]

vectors = np.array([model.wv[w] for w in words])

projection = TSNE(
    n_components=2,
    random_state=42,
    perplexity=min(5, len(words)-1)
).fit_transform(vectors)

plt.figure(figsize=(8,6))

for i, word in enumerate(words):
    plt.scatter(projection[i,0], projection[i,1])
    plt.text(projection[i,0], projection[i,1], word)

plt.title("Word2Vec Embeddings")
plt.show()



# 10. Using spaCy Word Vectors

Many NLP applications use pre-trained embeddings instead of training from scratch.

Install:

```bash
pip install spacy
python -m spacy download en_core_web_md
```


In [ ]:

import spacy

nlp = spacy.load("en_core_web_md")

anne = nlp("anne")
girl = nlp("girl")
school = nlp("school")

print("anne-girl similarity:", anne.similarity(girl))
print("anne-school similarity:", anne.similarity(school))


In [ ]:

token = nlp.vocab["teacher"]

print(token.vector.shape)
print(token.vector[:10])



# 11. Key Takeaways

1. Word2Vec learns dense vector representations of words.
2. The training signal comes from neighboring words.
3. Skip-Gram predicts context from a center word.
4. CBOW predicts the center word from context.
5. The embedding layer can be viewed as a latent representation similar to the encoder output in autoencoders.
6. After training, we keep the encoder (embeddings) and discard the decoder.
7. Pre-trained embeddings from spaCy and Gensim are widely used in real NLP systems.
8. Word2Vec was an important step toward modern language models and transformers.
